In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Fanasty value will be decided by ESPN fantasy scoring metrics. Rebounds & Points = 1 fantasy pt, 3-pointer = 1 pt, FGM = 2 pts, FTM = 1 pts, Assists = 2 pts, Steals & Blocks = 4 pts, FGA = -1 pts, FTA = -1 pts, & turnovers = -2 pts. ")
df = pd.read_csv("Player_Totals.csv", index_col = 0)
df.columns = df.columns.str.strip().str.lower()

In [ ]:
#print(df.head())df = df[df['season'] >= 2020]
df = df[df['pts'] >= 500] #filtering out players with less than 500 points because bench players or minimal impact role players aren't important.
columns_drop = ["orb","drb","mp","gs"] #unncessary data
df_fantasy = df.drop(columns=columns_drop + ["birth_year", "lg"], errors='ignore') 
def calc_fantasy_points(row):
    # Total field goals made = 2PM + 3PM, so 2PM = FG - 3Pdf['season_weight'] = 2.5 ** (df['season'] - 2020)
    df['weighted_fp'] = df['fantasy_ppg'] * df['season_weight']
player_weighted = df.groupby('player').agg({
    'weighted_fp': 'sum',
    'season_weight': 'sum'
}).reset_index()
player_weighted['weighted_fantasy_ppg'] = player_weighted['weighted_fp'] / player_weighted['season_weight']


In [ ]:

#adding a column for players who are a riskier pick than others
df_recent = df[df['season'].isin([2024, 2025])]

# Calculate total games played in 2024 + 2025 per player
recent_games = df_recent.groupby('player')['g'].sum().reset_index()
recent_games.rename(columns={'g': 'games_last_two_seasons'}, inplace=True)


df_fantasy = df_fantasy.merge(recent_games, on='player', how='left')
df_fantasy['games_last_two_seasons'] = df_fantasy['games_last_two_seasons'].fillna(0)


df_fantasy['durability'] = pd.cut(
    df_fantasy['games_last_two_seasons'],
    bins=[-1, 40, 100, 140, float('inf')],
    labels=['Very Risky', 'Risky', 'Moderate', 'Reliable']
) 
df_fantasy = df.drop(columns=columns_drop + ["birth_year", "lg"], errors='ignore')

In [ ]:
def calc_fantasy_points(row):
    twopm = row['fg'] - row['x3p']
    return (
        row['x3p'] * 5 +      
        twopm * 3 +           
        row['ft'] * 1 +       
        (row['fga'] + row['fta']) * -1 +  
        row['trb'] * 1 +      
        row['ast'] * 2 +      
        row['stl'] * 4 +      
        row['blk'] * 4 +      
        row['tov'] * -2       
    )


df['fantasy_points'] = df.apply(calc_fantasy_points, axis=1)
df['fantasy_ppg'] = df['fantasy_points'] / df['g']
df_fantasy['fantasy_points'] = df['fantasy_points']
df_fantasy['fantasy_ppg'] = df['fantasy_ppg']

In [ ]:
df_fantasy = df_fantasy.sort_values(by='fantasy_points', ascending=False)
df_fantasy = df.groupby('player').agg({
    'fantasy_ppg': 'mean',
    'fantasy_points': 'sum',
    'pts': 'mean',
    'trb': 'mean',
    'ast': 'mean',
    'stl': 'mean',
    'blk': 'mean',
    'tov': 'mean',
    'fg_percent': 'mean',
    'g': 'sum',
    'season': 'count'
}).rename(columns={'season': 'num_seasons'}).reset_index()
player_pos = df[['player', 'pos']].drop_duplicates(subset='player')
df_fantasy = df_fantasy.merge(player_pos, on='player', how='left')

df_fantasy = df_fantasy.merge(
    player_weighted[['player', 'weighted_fantasy_ppg']],
    on='player',
    how='left'
)


cols = ['player', 'weighted_fantasy_ppg'] + [col for col in df_fantasy.columns if col not in ['player', 'weighted_fantasy_ppg']]
df_fantasy = df_fantasy[cols]
df_fantasy = df_fantasy[df_fantasy['g'] >= 120]

df_fantasy = df_fantasy.sort_values(by='weighted_fantasy_ppg', ascending=False)

In [ ]:
while True:
    print("\n NBA Fantasy Console Menu")
    print("1. Show Top 100 Fantasy Players (Weighted Fantasy PPG)")
    print("2. Show Top 20 Fantasy Players by Position")
    print("3. Show Advanced Metrics for a Specific Player")
    print("4. Exit")
    
    choice = input("Enter your choice (1–4): ").strip()

    if choice == '1':
        top100 = df_fantasy.sort_values(by='weighted_fantasy_ppg', ascending=False).head(100)
        print("\nTop 100 NBA Fantasy Players:")
        print(top100[['player', 'pos', 'weighted_fantasy_ppg', 'fantasy_ppg', 'durability']])

    elif choice == '2':
        pos_input = input("Enter a position (e.g., PG, SG, SF, PF, C): ").strip().upper()
        top_pos = df_fantasy[df_fantasy['pos'] == pos_input].sort_values(by='weighted_fantasy_ppg', ascending=False).head(20)
        
        if top_pos.empty:
            print(f"No players found for position '{pos_input}'.")
        else:
            print(f"\nTop 20 {pos_input}s by Weighted Fantasy PPG:")
            print(top_pos[['player', 'pos', 'weighted_fantasy_ppg', 'fantasy_ppg', 'durability']])

    elif choice == '3':
        name_input = input("Enter the full or partial name of the player: ").strip().lower()
        matches = df[df['player'].str.lower().str.contains(name_input)]

        if matches.empty:
            print(f"No players found matching '{name_input}'.")
        else:
            selected_player = matches.iloc[0]['player']
            player_row = matches[matches['player'] == selected_player]

            print(f"\nAdvanced Stats for {selected_player} (per game averages shown):")
            player_per_game = {
                'PPG': (player_row['pts'] / player_row['g']).mean(),
                'RPG': (player_row['trb'] / player_row['g']).mean(),
                'APG': (player_row['ast'] / player_row['g']).mean(),
                'SPG': (player_row['stl'] / player_row['g']).mean(),
                'BPG': (player_row['blk'] / player_row['g']).mean(),
                'TOPG': (player_row['tov'] / player_row['g']).mean(),
                'FG%': player_row['fg_percent'].mean(),
                '3P%': (player_row['x3p_percent']* 100).mean(),
                '2P%': player_row['x2p_percent'].mean(),
                'eFG%': player_row['e_fg_percent'].mean()
            }

            for stat, val in player_per_game.items():
                print(f"{stat}: {val:.2f}")

            #setting up the plot
            stat_mapping = {
                'PPG': df['pts'] / df['g'],
                'RPG': df['trb'] / df['g'],
                'APG': df['ast'] / df['g'],
                'SPG': df['stl'] / df['g'],
                'BPG': df['blk'] / df['g'],
                'TOPG': df['tov'] / df['g'],
                'FG%': df['fg_percent'],
                '3P%': df['x3p_percent'] * 100,
                '2P%': df['x2p_percent'],
                'eFG%': df['e_fg_percent']
            }

            fig, axes = plt.subplots(4, 3, figsize=(18, 16))  # 4 rows x 3 cols gives 12 slots
            axes = axes.flatten()


            for i, (stat_name, series) in enumerate(stat_mapping.items()):
                sns.histplot(series, kde=True, ax=axes[i], color='skyblue')
                axes[i].axvline(player_per_game[stat_name], color='red', linestyle='--', label=selected_player)
                axes[i].set_title(f"{stat_name} Distribution")
                axes[i].legend()

            # Hide any unused subplots
            for j in range(i+1, len(axes)):
                axes[j].set_visible(False)

            plt.tight_layout()
            plt.show()




    elif choice == '4':
        print("Exiting NBA Fantasy Console. Have a great season!")
        break

    else:
        print("Invalid choice. Please enter 1, 2, 3, or 4.")